In [ ]:
import tonic
import tonic.transforms as T
import numpy as np
import matplotlib.pyplot as plt

dataset = tonic.datasets.NMNIST(save_to="./data", train=False) # time, number of polarities, height, width

sensor_size = dataset.sensor_size
print("Sensor size:", sensor_size) #x, y, number of polarities
events, target = dataset[0]
print(events[0])
events_duration_microsecond = events[-1][2]
print(f"total duration (ms): {events_duration_microsecond * 10 ** -3}, approx. events per ms: {len(events) * 10 ** 3 / events_duration_microsecond}") # time bins are in microseconds
frame_transform = T.ToFrame(sensor_size=sensor_size, time_window=50 * 10 ** 3) # 50ms time window

frames = frame_transform(events)
print("frames shape:", frames.shape) # number of time bins, number of polarities, height, width

In [ ]:
# increase canvas size, anchoring to the center
def resize_canvas(frames, new_size):
    """Increase canvas size, anchoring to the center"""
    # Get dimensions
    num_frames, num_polarities, width, height = frames.shape
    pad_width_before = (new_size - width) // 2
    pad_width_after = new_size - width - pad_width_before
    # Calculate padding (handle both odd and even cases properly)
    pad_height_before = (new_size - height) // 2
    pad_height_after = new_size - height - pad_height_before
    
    
    # Create padding tuple for all dimensions
    pad_tuple = ((0, 0),                               # frames dimension
                 (0, 0),                               # polarities dimension
                 (pad_height_before, pad_height_after), # height dimension
                 (pad_width_before, pad_width_after))   # width dimension
    
    # Apply padding to all frames at once
    new_frames = np.pad(frames, pad_tuple, mode='constant', constant_values=0)
    
    return new_frames

In [ ]:
def calculate_center_of_gravity_combined_events(frames, events):
    # events are in the format (x, y, time)
    # we want to calculate the cumulative center of gravity for each time bin
    # shape is frame_n, polarity, x, y
    # returns cog per frame as a tuple of [(x, y)]
    
    # infer time window from number of frames
    time_window = events[-1][2] / frames.shape[0]
    print(f"I infer the time window as {time_window * 10 ** -3} ms")
    
    frame_cogs = []
    for i, frame in enumerate(frames):
        start_time = i * time_window
        end_time = (i + 1) * time_window
        
        # Filter events within the current time window
        relevant_events = [e for e in events if start_time <= e[2] < end_time]
        total_x = sum(e[0] for e in relevant_events)
        total_y = sum(e[1] for e in relevant_events)
        count = len(relevant_events)

        cog_x = total_x / count
        cog_y = total_y / count

        frame_cogs.append((cog_x, cog_y))
    
    return frame_cogs        

In [ ]:
def calculate_bounding_squares(frames, events, percentage=90):
    """
    Calculate a square that has the center of gravity as its center and 
    encompasses at least x% of the events in each time bin.
    Returns:
        List of tuples (cog_x, cog_y, side_length, corners) for each frame
    """
    cogs = calculate_center_of_gravity_combined_events(frames, events)
    time_window = events[-1][2] / frames.shape[0]
    
    bounding_squares = []
    for i, (cog_x, cog_y) in enumerate(cogs):
        start_time = i * time_window
        end_time = (i + 1) * time_window
        relevant_events = [e for e in events if start_time <= e[2] < end_time]

        # Calculate the distance from each event to the CoG in terms of square boundary
        # For a square, we need the maximum of |x-cog_x| and |y-cog_y| for each event
        distances = [max(abs(e[0] - cog_x), abs(e[1] - cog_y)) for e in relevant_events]
        distances.sort()
        min_events_to_include = max(1, int(len(relevant_events) * percentage / 100))
        index = min(min_events_to_include - 1, len(distances) - 1)
        required_distance = distances[index]
        side_length = 2 * required_distance
        corners = get_corners(cog_x, cog_y, side_length)
        bounding_squares.append((cog_x, cog_y, side_length, corners))
    
    return bounding_squares

def get_corners(center_x, center_y, side_length):
    """Calculate the corners of a square given its center and side length"""
    half_side = side_length / 2
    top_left = (center_x - half_side, center_y - half_side)
    top_right = (center_x + half_side, center_y - half_side)
    bottom_left = (center_x - half_side, center_y + half_side)
    bottom_right = (center_x + half_side, center_y + half_side)
    return [top_left, top_right, bottom_right, bottom_left]

In [ ]:
def combine_polarities(frames):
    # frames are of shape time_frame, polarity, height, width
    # we can concat both polarities (there might be some overlap, but that's ok!)
    # negative polarities are turned into negative numbers
    polarity_zero = -frames[:, 0, :, :]
    polarity_one = frames[:, 1, :, :]
    # Take the value with the largest magnitude
    mask = np.abs(polarity_zero) > np.abs(polarity_one)
    combined = np.where(mask, polarity_zero, polarity_one)
    
    # print(combined.shape)
    return combined


In [ ]:
def plot_frames(frames):
    n_frames = len(frames)
    fig, axes = plt.subplots(2, n_frames, figsize=(n_frames * 3, 6))
    
    for i, frame in enumerate(frames):
        # frame[0] is polarity 0 and frame[1] is polarity 1
        axes[0, i].imshow(frame[1], cmap='gray')
        axes[0, i].set_title("Polarity OFF (0)")
        axes[1, i].imshow(frame[0], cmap='gray')
        axes[1, i].set_title("Polarity ON (1)")
    plt.tight_layout()
    plt.show()

def plot_combined_frames(combined_frames, bounding_squares):
    n_frames = len(combined_frames)
    h, w = combined_frames[0].shape
    _, ax = plt.subplots(1, n_frames, figsize=(n_frames * 3, 3))
    for i, combined_frame in enumerate(combined_frames):
        ax[i].imshow(combined_frame, cmap='gray')
        ax[i].set_title(f"Combined Polarity {i}")
        ax[i].axis('off')
        cog_x, cog_y, side_length, corners = bounding_squares[i]

        ax[i].plot(cog_x, cog_y, 'ro', markersize=4)
        rect = plt.Rectangle((cog_x - side_length // 2, cog_y - side_length // 2), 
                            side_length, side_length, linewidth=2, edgecolor='r', facecolor='none')
        ax[i].add_patch(rect)
    plt.show()

In [ ]:
# before resizing
# plot_frames(frames)
combined_original_frames = combine_polarities(frames)
original_frame_cogs = calculate_center_of_gravity_combined_events(frames, events)
print("original frame cogs:", original_frame_cogs)
# original numbers
original_side = sensor_size[0]
basic_bounding_squares = [(cog[0], cog[1], original_side // 2, original_side // 2) for cog in original_frame_cogs]
plot_combined_frames(combined_original_frames, basic_bounding_squares)

# fancy bounding squares
bounding_squares = calculate_bounding_squares(frames, events, percentage=95)
plot_combined_frames(combined_original_frames, bounding_squares)

In [ ]:
def calculate_resized_cogs(new_size, old_size, cogs):
    # everything is in the center of the new canvas
    size_diff = new_size - old_size
    new_cogs = []
    for cog in cogs:
        x, y = cog
        new_x = x + size_diff // 2
        new_y = y + size_diff // 2
        new_cogs.append((new_x, new_y))
    return new_cogs

In [ ]:
# original size is 34x34, we want to increase it to e.g. 68x68
new_size = 68
frames = resize_canvas(frames, new_size)
# plot_frames(frames)
combined_resize_frames = combine_polarities(frames)
resize_frame_cogs = calculate_resized_cogs(new_size, original_side, original_frame_cogs)

basic_bounding_squares = [(cog[0], cog[1], new_size // 2, new_size // 2) for cog in resize_frame_cogs]
plot_combined_frames(combined_resize_frames, basic_bounding_squares)